# Reproducing Mnemosyne (SIGMOD '25) on Chameleon

**Paper:** *Mnemosyne: Dynamic Workload-Aware BF Tuning via Accurate Statistics in LSM trees*
Zichen Zhu, Yanpeng Wei, Ju Hyoung Mun, Manos Athanassoulis. SIGMOD 2025. https://doi.org/10.1145/3725327

**Original code:** https://github.com/BU-DiSC/Mnemosyne

This notebook reproduces **Figure 12**, the paper's main evaluation: **read bytes** (panels a–f) and **read latency** (panels g–l) of **RocksDB**, **Mnemosyne** and **Mnemosyne+**, for different bits-per-key, with 0%, 50% and 100% empty queries (Z), under uniform and normal query distributions.

**Smaller setup, so it finishes in about 5–6 hours** (it uses the authors' own scripts, with only the sizes changed):

| | Paper | This notebook |
|---|---|---|
| Inserts | 21M | 2.625M (⅛) |
| Queries / updates | 31M / 10M | 1.94M / 0.625M (1/16, same ratio) |
| Block cache | 512 MB | 64 MB (⅛, same cache-to-data ratio) |
| Bits-per-key | 2, 3, 4, 5, 6, 7 | 2, 4, 6 |
| Runs per setting | 1 | 1 |

Because the data is smaller and the hardware is different, the absolute numbers differ from the paper. The thing to compare is the **trend**: Mnemosyne and Mnemosyne+ should read fewer bytes and have lower read latency than RocksDB, especially at low bits-per-key and with normal (skewed) queries.

Steps: reserve a bare-metal machine → start a server → connect by SSH → clone the forked repo → run `automate.sh` → show the figure.

# Create lease
1. Choose site
2. Select a project that you are member of
3. Specify a lease name that you are not already using in this site
4. Choose the desired properties (e.g., "compute_skylake") of the baremetal machine
5. Verify that the lease was successful

In [ ]:
import chi
chi.use_site('CHI@TACC')

PROJECT_name = "CHI-261568"   # change this to your own Chameleon project
chi.set("project_name", PROJECT_name)
print(f'Using Project {chi.get("project_name")}')

Optional: see which machine types are free right now at this site. If the lease cell says "not enough resources", pick a type from this list that has at least 1 free machine.


In [ ]:
from chi import hardware
from collections import Counter

free = hardware.get_nodes(filter_reserved=True)
print(Counter(n.type for n in free))


In [ ]:
from chi import lease

import time
import keystoneauth1, blazarclient
from chi import lease

reservations = []
LEASE_KEY = 'reproduce-sigmod25-mnemosyne'
lease_node_type = "compute_skylake"        # pick a type that is free (see the cell above)
try:
    lease_obj=lease.get_lease(LEASE_KEY)
except Exception as e:
    lease_obj=None
if lease_obj!=None:
    print("lease already created")
    print(lease_obj.node_reservations[0]["id"])
else:
    try:
        print(f"Creating lease with name = {LEASE_KEY}...")
        lease.add_fip_reservation(reservations, count=1)
        lease.add_node_reservation(reservations, count=1, node_type=lease_node_type)

        start_date, end_date = lease.lease_duration(hours=14, days=0)

        l = lease.create_lease(
            LEASE_KEY, 
            reservations, 
            start_date=start_date, 
            end_date=end_date
        )
        lease_id = l["id"]
        lease_obj=lease.get_lease(LEASE_KEY)

        print("Waiting for lease to start ...")
        lease_obj.wait()
        print("Lease is now active!")
    except blazarclient.exception.BlazarClientException as e:
        print(f"There is an issue making the reservation. Please check the host calendar.")
        print("https://chi.tacc.chameleoncloud.org/project/leases/calendar/host/")
        print(e)
    except Exception as e:
        print("An unexpected error happened.")
        print(e)

In [ ]:
print(lease_obj.node_reservations[0]["id"])
print(lease_obj)

# Create server and assign ip

In [ ]:
from chi import server
server.list_servers()

image = "CC-Ubuntu24.04"
LEASE_KEY = 'reproduce-sigmod25-mnemosyne'
try:
    s=server.get_server(LEASE_KEY)
except Exception as e:
    s=None

if s!=None:
    print("server already started")
else:
    try:
        s = server.create_server(
            LEASE_KEY, 
            image_name=image,
            flavor_name="baremetal",
            reservation_id=lease_obj.node_reservations[0]["id"]
        )
    except Exception as e:
        print("An unexpected error happened.")
        print(e)

    print("Waiting for server to start ...")
    server.wait_for_active(s.id)
    print("Done")

In [ ]:
server.get_server('reproduce-sigmod25-mnemosyne').status

In [ ]:
import chi
from chi import lease

server_id = chi.server.get_server_id(LEASE_KEY)
print(server_id)

floating_ip = lease_obj.get_reserved_floating_ips()
print(floating_ip)
floating_ip = floating_ip[0]
try:
    ip=server.get_floating_ip()
except Exception as e:
    ip=None
if ip != None:
    print("server already get the ip")
else:
    server.associate_floating_ip(server_id, floating_ip_address=floating_ip)
ip = floating_ip
print(ip)

In [ ]:
print(f"Server is accessible at IP address: {ip}")
chi.server.wait_for_tcp(ip, port=22) # Wait for SSH to be ready

# Create SSH connection
All commands are passed through this connection

In [ ]:
import chi
from chi.ssh import Remote
import paramiko

key_path = "/work/.ssh/id_rsa"

# Load your private key
pkey = paramiko.RSAKey.from_private_key_file(key_path)

conn = Remote(
    ip=ip,
    user="cc",
    connect_kwargs={
        "pkey": pkey,
        "key_filename": [],
        "allow_agent": False,
        "look_for_keys": False,
    },
)

conn.run("hostname")

# Check properties of the newly created instance

In [ ]:
conn.run('free -h')
conn.run('df -h')
conn.run('nproc')

# Clone the repository for experiment reproducibility (forked from the original)

In [ ]:
conn.run('git clone https://github.com/aroma058/trovi_reproduce_mnemosyne-sigmod25-fork.git')

In [ ]:
conn.run('ls -al trovi_reproduce_mnemosyne-sigmod25-fork')

# Run the script that reproduces Figure 12
`automate.sh` does all of this:
1. Installs dependencies
2. Downloads the code the paper needs (RocksDB fork, benchmark, workload generator)
3. Makes the experiment smaller (⅛ of the data and cache, 1/16 of the operations, bits-per-key 2, 4, 6)
4. Compiles everything
5. Runs the authors' `fig12.sh`: makes the workloads, then runs 18 tests (6 workloads × 3 bits-per-key), each comparing RocksDB, Mnemosyne and Mnemosyne+
6. Joins the 12 panels into one picture: `exp-figures/Fig12.png`

This takes about 5–6 hours (5.5 hours on a `compute_skylake` node), so it runs in the background. The output goes to `run.log`.

In [ ]:
conn.run('cd trovi_reproduce_mnemosyne-sigmod25-fork && nohup bash automate.sh > run.log 2>&1 < /dev/null &', disown=True)
print("Started. Use the next cell to check progress.")

# Check progress
Run this cell again any time. When it is finished, the last line says **Done. Figure saved at exp-figures/Fig12.png**.

If the notebook was restarted or the connection dropped, run the project, server, IP and SSH cells again first (they will say "already created" / "already started").

In [ ]:
conn.run('tail -n 15 trovi_reproduce_mnemosyne-sigmod25-fork/run.log')

How many of the 18 tests are finished:

In [ ]:
conn.run('ls trovi_reproduce_mnemosyne-sigmod25-fork/skew-aware-bpk-benchmark/output-bpk*/*-output.txt 2>/dev/null | wc -l')

# Show the result
`automate.sh` saved our Figure 12 as one picture (`Fig12.png`) and the authors' Figure 12 from their repository as `Fig12_original.png`.

Layout: rows are read bytes (uniform, normal) and read latency (uniform, normal); columns are Z = 0.0, 0.5, 1.0. Black is RocksDB, blue is Mnemosyne, red is Mnemosyne+.

In [ ]:
conn.get('trovi_reproduce_mnemosyne-sigmod25-fork/exp-figures/Fig12.png', 'Fig12.png')
conn.get('trovi_reproduce_mnemosyne-sigmod25-fork/exp-figures/Fig12_original.png', 'Fig12_original.png')

from IPython.display import Image, display
print("Our result (smaller setup):")
display(Image(filename='Fig12.png'))

In [ ]:
print("Authors' Figure 12 (full setup):")
display(Image(filename='Fig12_original.png'))

**What to look for.** RocksDB (black) should be highest in every panel, with the biggest gap at low bits-per-key. Mnemosyne+ (red) should be clearly below Mnemosyne (blue) for normal (skewed) queries, especially at Z = 1.0.

**Expected differences from the paper.** Read bytes are about 15× smaller, because this run uses 1/16 of the operations. Read latency is about 5× higher, because the Chameleon node's SSD is slower than the authors' Optane SSD. These change the numbers, not the trends.

How long the run took:

In [ ]:
conn.run('grep -E "Started at|Finished at" trovi_reproduce_mnemosyne-sigmod25-fork/run.log')

# Save the raw results
Copies the measured numbers to your Jupyter folder, so you keep them after the lease ends.

In [ ]:
conn.run('cd trovi_reproduce_mnemosyne-sigmod25-fork/skew-aware-bpk-benchmark && tar czf ~/fig12_results.tar.gz output-bpk* agg_output')
conn.get('fig12_results.tar.gz', 'fig12_results.tar.gz')

# Clean up
Delete the lease when you are done, so the machine is free for others. Deleting the lease also deletes the server.

In [ ]:
from chi import lease
LEASE_KEY = 'reproduce-sigmod25-mnemosyne'
lease.delete_lease(lease.get_lease(LEASE_KEY).id)
print("Lease deleted.")
